<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

## Apply KKT to a Linear Program

**Express an error limit as a linear program and check its solution with KKT conditions.**

We keep the recorded clock errors and use one constant correction. The objective now compares the largest absolute error instead of the sum of squared errors.
The rate setting, calibration equality, and initial-correction bound belong to the previous formulation and are not imposed here.

Introducing a limit on every absolute error gives a **linear program (LP)**. LP describes the form of the objective and constraints; KKT supplies conditions for its solution. Here, *programming* means choosing a plan.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 1 · Compare the largest errors first

Keep the recorded errors $-2,-1,-1,0$ min. Choose a real correction $q$, in minutes, and add it to every reading.
An **absolute error** is the error's size without its sign: both $-2$ and $2$ have size 2.

| Correction | Remaining errors (min) | Largest absolute error (min) |
|:---|:---|---:|
| $q=0$ | $-2,-1,-1,0$ | $2$ |
| $q=1$ | $-1,0,0,1$ | $1$ |
| $q=2$ | $0,1,1,2$ | $2$ |

The middle choice is best among these three. We will show why no other real correction can do better.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

Start by calculating the errors. `np.abs` removes their signs and `np.max` selects the largest size.


In [ ]:
import numpy as np

OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min


def simulate_clock(correction, observed_errors=OBSERVED_ERRORS):
    return np.asarray(observed_errors) + correction


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

Compare the three corrections from the table using the same response calculation.


In [ ]:
for correction in (0.0, 1.0, 2.0):
    remaining_errors = simulate_clock(correction)
    largest_error = float(np.max(np.abs(remaining_errors)))
    print(f"q = {correction:.1f} min | errors = {remaining_errors} min"
          f" | largest error = {largest_error:.1f} min")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 2 · Write the error limits

Let $z$ be an upper limit on every absolute error, in minutes. It is an extra variable in the calculation; $q$ remains the physical clock setting.
For example, $z=1$ requires every remaining error to lie between $-1$ and $1$ min.

Hold the proposed limit at $z=1$ min and change only the correction. At $q=0$, one error lies outside the shaded interval. At $q=1$, all four fit. Errors on a boundary are allowed; stacked dots are separate observations with the same error.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-1_constrained_optimization_kkt/assets/clock_error_limits.png" alt="With allowed error interval minus 1 to plus 1 minutes, correction zero is rejected because the minus 2 minute error is outside. Correction 1 minute is accepted because errors minus 1, 0, 0, plus 1 are all inside or on the interval boundaries." width="800" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

We minimize $z$ while keeping all four errors within its limits:

$$
\begin{aligned}
\underset{q,z\in\mathbb R}{\operatorname{minimize}}\quad &z\\
\text{subject to}\quad &-z\le q-2\le z,\\
&-z\le q-1\le z,\\
&-z\le q\le z,\\
&z\ge0.
\end{aligned}
$$

The middle inequality covers both recorded errors of $-1$ min. The objective and inequalities contain only linear expressions plus constants, so this is an LP.
There are no squares or products of decision variables in this formulation.

For a fixed $q$, the smallest allowed $z$ is exactly the largest absolute error. Minimizing $z$ therefore asks the intended question.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

Check whether a proposed correction and error limit satisfy every requirement. The evaluation keeps feasibility separate from the objective, which is the proposed limit.


In [ ]:
def evaluate_limit(correction, error_limit, observed_errors=OBSERVED_ERRORS):
    remaining_errors = simulate_clock(correction, observed_errors)
    largest_error = float(np.max(np.abs(remaining_errors)))
    feasible = bool(error_limit >= 0 and np.all(np.abs(remaining_errors) <= error_limit))
    return {
        "decision": correction,
        "response": remaining_errors,
        "largest_error": largest_error,
        "objective": error_limit,
        "feasible": feasible,
    }


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

Compare $(q,z)=(0,2),(0,1),(1,1),(1,0.5)$. A smaller proposed limit is useful only if all errors fit inside it. At $q=1$, reducing $z$ from 1 to 0.5 min leaves the errors $-1,0,0,1$ min unchanged. The errors $-1$ and $1$ then violate the limits, so $(1,0.5)$ is rejected despite its smaller objective value.


In [ ]:
acceptable = []
for correction, error_limit in ((0.0, 2.0), (0.0, 1.0), (1.0, 1.0), (1.0, 0.5)):
    result = evaluate_limit(correction, error_limit)
    if not result["feasible"]:
        print(f"q = {correction:g}, z = {error_limit:g}: reject; an error exceeds the limit")
        continue
    acceptable.append(result)
    print(f"q = {correction:g}, z = {error_limit:g}: acceptable")

best_listed = min(acceptable, key=lambda result: result["objective"])
print("Best acceptable listed pair:", (best_listed["decision"], best_listed["objective"]))


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

This comparison selects the best acceptable pair in the list. The next argument proves the result for every real correction.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 3 · Find the smallest possible limit

The first and last remaining errors are $q-2$ and $q$. Their difference is always 2 min.
An allowed interval from $-z$ to $z$ has width $2z$. To fit both errors, it must be at least 2 min wide, so $z\ge1$.

At $q=1$, all remaining errors are $-1,0,0,1$ min. They satisfy the limits with $z=1$.
This reaches the smallest possible limit and proves optimality.

This example selects the same correction as least squares, but measures a different result:
the largest absolute error is 1 min; the sum of squared errors is 2 min². With other observations, the rules can select different corrections.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

The smallest and largest recorded errors are $-2$ and $0$ min. Their midpoint is $-1$ min.
Adding 1 min centers those two extremes around zero. The best limit is half their separation.
The code uses this direct calculation, then checks all four errors.


In [ ]:
smallest_observation = float(np.min(OBSERVED_ERRORS))
largest_observation = float(np.max(OBSERVED_ERRORS))
best_correction = -(smallest_observation + largest_observation) / 2
best_limit = (largest_observation - smallest_observation) / 2
best_result = evaluate_limit(best_correction, best_limit)
print("Best correction:", best_correction, "min")
print("Best limit:", best_limit, "min")
print("Remaining errors:", best_result["response"], "min")
print("All requirements satisfied:", best_result["feasible"])


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 4 · Check the LP solution with KKT

At $(q,z)=(1,1)$, only two of the displayed error inequalities are active:

$$
2-q-z\le0,\qquad q-z\le0.
$$

They limit the first reading's negative error and the last reading's positive error.
The other error inequalities and $-z\le0$ are strict, so their multipliers are zero.
Call the two active multipliers $\alpha$ and $\beta$. The relevant Lagrangian terms are

$$
L(q,z,\alpha,\beta)=z+\alpha(2-q-z)+\beta(q-z).
$$

Stationarity gives

$$
\frac{\partial L}{\partial q}=-\alpha+\beta=0,\qquad
\frac{\partial L}{\partial z}=1-\alpha-\beta=0.
$$

Thus $\alpha=\beta=0.5$. Both are nonnegative, the candidate is feasible, and every multiplier times its constraint residual is zero.
This verifies the KKT conditions. Section 3 already proved that no smaller limit is possible.

The code includes all seven distinct inequality residuals, so inactive constraints are checked too. The repeated observation of $-1$ min uses the same pair of inequalities.


In [ ]:
correction, error_limit = 1.0, 1.0
q, z = correction, error_limit
inequality_names = ["first lower", "first upper", "middle lower", "middle upper",
                    "last lower", "last upper", "nonnegative limit"]
residuals = np.array([2-q-z, q-2-z, 1-q-z, q-1-z, -q-z, q-z, -z])
multipliers = np.array([0.5, 0, 0, 0, 0, 0.5, 0])
constraint_gradients = np.array([[-1, -1], [1, -1], [-1, -1], [1, -1],
                                 [-1, -1], [1, -1], [0, -1]])
objective_gradient = np.array([0.0, 1.0])  # objective is z
stationarity = objective_gradient + multipliers @ constraint_gradients
for name, residual, multiplier in zip(inequality_names, residuals, multipliers):
    print(f"{name}: residual = {residual:g}, multiplier = {multiplier:g}")
print("Feasible:", bool(np.all(residuals <= 0)))
print("Nonnegative multipliers:", bool(np.all(multipliers >= 0)))
print("Complementarity:", bool(np.allclose(multipliers * residuals, 0)))
print("Stationarity:", stationarity)


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

The largest-error LP formulation is described in [Boyd and Vandenberghe, Section 6.1.1](https://web.stanford.edu/~boyd/cvxbook/bv_cvxbook.pdf).


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 5 · See why the comparison rule matters

Change only the last recorded error from 0 to 3 min, giving $-2,-1,-1,3$ min. Retain one constant correction and no calibration or correction bound. Hold these changed observations fixed while comparing least squares with the largest-error rule.

Least squares cancels the mean, giving $q=0.25$ min. Minimizing the largest error centers the extremes $-2$ and $3$, giving $q=-0.5$ min.
The first choice gives squared-error score 14.75 min² and largest error 3.25 min.
The second gives squared-error score 17 min² and largest error 2.5 min.
Each wins under its own comparison rule.

For a fixed correction, changing the comparison rule leaves the remaining errors unchanged. The two scores have different units, so compare candidates using the same rule.


In [ ]:
changed_errors = OBSERVED_ERRORS.copy()
changed_errors[-1] = 3.0
least_squares_correction = -float(np.mean(changed_errors))
largest_error_correction = -float(np.min(changed_errors) + np.max(changed_errors)) / 2

for name, correction in (("Least squares", least_squares_correction),
                         ("Largest error", largest_error_correction)):
    remaining_errors = simulate_clock(correction, changed_errors)
    squared_score = float(np.sum(remaining_errors ** 2))
    largest_error = float(np.max(np.abs(remaining_errors)))
    print(f"{name}: q = {correction:.2f} min"
          f" | squares = {squared_score:.2f} min² | largest = {largest_error:.2f} min")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

This LP and the preceding QP use the same four KKT conditions. Their objective forms differ, but feasibility, multiplier signs, complementarity, and stationarity retain their meanings.
